## Build 24h Nursing Notes per ICU Stay (NOTEEVENTS → notes_24h_nursing.parquet)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/drive/MyDrive/mimic_data"
OUT_DIR  = "/content/drive/MyDrive/mimic_outputs"

COHORT_PATH = os.path.join(OUT_DIR, "cohort_icustay.parquet")
NOTE_PATH   = os.path.join(DATA_DIR, "noteevents_nursing_subset.parquet")

In [3]:
import re

# ----------------------------
# Config (optional safeguards)
# ----------------------------
MAX_CHARS = 20000          # cap very long concatenated notes (helps ClinicalBERT runtime)
REMOVE_PHI = True          # remove MIMIC-III safe-harbor placeholders like [** ... **]
PHI_TOKEN = " PHI_TOKEN "  # replacement token

PHI_RE = re.compile(r"\[\*\*.*?\*\*\]")
WS_RE = re.compile(r"\s+")

def clean_note_text(text: str) -> str:
    """Lightweight text cleaning for clinical notes (keeps clinical content intact)."""
    if text is None:
        return ""
    s = str(text)
    if REMOVE_PHI:
        s = PHI_RE.sub(PHI_TOKEN, s)
    s = WS_RE.sub(" ", s).strip()
    return s

# 1) Load ICU-stay anchors (HADM_ID / ICUSTAY_ID / INTIME)
cohort = pd.read_parquet(COHORT_PATH)[["hadm_id", "icustay_id", "intime"]].copy()
cohort["intime"] = pd.to_datetime(cohort["intime"], errors="coerce")
cohort = cohort.dropna(subset=["icustay_id", "hadm_id", "intime"])
cohort["hadm_id"] = cohort["hadm_id"].astype(int)
cohort["icustay_id"] = cohort["icustay_id"].astype(int)

# 2) Load nursing notes subset (already filtered to Nursing/other)
notes = pd.read_parquet(NOTE_PATH)

# Use CHARTTIME when available; fall back to CHARTDATE (midnight) if CHARTTIME is missing
notes["charttime"] = pd.to_datetime(notes.get("charttime"), errors="coerce")
notes["chartdate"] = pd.to_datetime(notes.get("chartdate"), errors="coerce")
notes["note_time"] = notes["charttime"].fillna(notes["chartdate"])

# Keep only notes with valid keys/timestamps/text
notes = notes.dropna(subset=["hadm_id", "note_time", "text"])
notes["hadm_id"] = notes["hadm_id"].astype(int)

# Drop empty/whitespace-only notes
notes["text"] = notes["text"].astype(str)
notes = notes[notes["text"].str.strip() != ""].copy()

# Optional: lightweight text cleaning (PHI placeholders + whitespace normalization)
notes["text"] = notes["text"].apply(clean_note_text)
notes = notes[notes["text"].str.strip() != ""].copy()

# 3) Pre-group cohort by HADM_ID to obtain ICU admission windows per hospitalization
# Each ICU stay defines a 24h window: [intime, intime + 24h)
cohort_by_hadm = {
    hid: g[["icustay_id", "intime"]].sort_values("intime")
    for hid, g in cohort.groupby("hadm_id")
}

assigned_parts = []

# 4) Assign each note to an ICU stay window within the same HADM_ID
for hid, g_note in notes.groupby("hadm_id"):
    if hid not in cohort_by_hadm:
        continue

    win = cohort_by_hadm[hid]
    icu_ids = win["icustay_id"].to_numpy()
    starts = win["intime"].to_numpy(dtype="datetime64[ns]")
    ends = (win["intime"] + pd.Timedelta(hours=24)).to_numpy(dtype="datetime64[ns]")

    t = g_note["note_time"].to_numpy(dtype="datetime64[ns]")

    # Boolean matrix: note_time falls into the i-th ICU window
    mask = (t[:, None] >= starts[None, :]) & (t[:, None] < ends[None, :])

    # If a note matches multiple windows (rare), assign it to the closest ICU start time
    diff = (t[:, None] - starts[None, :]).astype("timedelta64[ns]").astype("int64")
    diff[~mask] = np.iinfo(np.int64).max

    best_idx = diff.argmin(axis=1)
    has_match = mask.any(axis=1)

    out = g_note.loc[has_match, ["text", "note_time"]].copy()
    out["icustay_id"] = icu_ids[best_idx[has_match]]
    assigned_parts.append(out[["icustay_id", "note_time", "text"]])

assigned = (
    pd.concat(assigned_parts, ignore_index=True)
    if assigned_parts
    else pd.DataFrame(columns=["icustay_id", "note_time", "text"])
)
print("assigned notes rows:", len(assigned), "unique icustay:", assigned["icustay_id"].nunique())

# 5) Aggregate: concatenate all notes within the 24h window per ICU stay (time-sorted)
assigned = assigned.sort_values(["icustay_id", "note_time"])
notes_24h = (
    assigned.groupby("icustay_id")["text"]
    .apply(lambda x: "\n\n".join(x.tolist()))
    .reset_index()
    .rename(columns={"text": "note_text_24h"})
)

# Optional: cap very long concatenated notes (helps embedding runtime/memory)
notes_24h["note_text_24h"] = notes_24h["note_text_24h"].astype(str).str.slice(0, MAX_CHARS)

out_path = os.path.join(OUT_DIR, "notes_24h_nursing.parquet")
notes_24h.to_parquet(out_path, index=False, compression="snappy")
print("✅ saved:", out_path, "shape:", notes_24h.shape)

assigned notes rows: 124249 unique icustay: 36691
✅ saved: /content/drive/MyDrive/mimic_outputs/notes_24h_nursing.parquet shape: (36691, 2)
